# 必要リソースの推定

この章では、`profile` サブコマンドで pipeline state(JSON) を解析し、必要リソースを見積もります。
比較対象は前章で生成した `Dim2` / `Dim3` / `DistributedDim2` / `PBC` の 4 ケースです。


この章で確認する内容は次のとおりです。

- `profile` は pipeline state JSON を入力に取ること
- `runtime`, `gate_count`, `code_distance`, `num_physical_qubits` を横比較すること
- PBC モードで値がどう変化するかを読むこと


In [1]:
import json
import pathlib
import os
import platform

from IPython.display import Code

project_root = pathlib.Path("../..").resolve()
if platform.system() == "Windows":
    qret_path = project_root / "build" / "bin"
    gridsynth_path = project_root / "externals" / "bin" / "gridsynth.exe"
elif platform.system() == "Darwin": 
    qret_path = project_root / "build" / "main"
    gridsynth_path = project_root / "externals" / "bin" / "gridsynth_macos"
elif platform.system() == "Linux":
    qret_path = project_root / "build" / "main"
    gridsynth_path = project_root / "externals" / "bin" / "gridsynth"
else:
    raise ValueError("Unsupported platform")
os.environ["GRIDSYNTH_PATH"] = str(gridsynth_path)
os.environ["PATH"] = str(qret_path) + os.pathsep + os.environ.get("PATH", "")


## 0. profile コマンド確認

In [2]:
!qret profile --help

qret 'profile' options:
  -h [ --help ]                         Show this help and exit.
  --quiet                               Suppress non-error output.
  --verbose                             Enable verbose logging (more detail 
                                        than default).
  --debug                               Enable debug logging (most detailed; 
                                        implies --verbose).
  --color                               Enable colored output.
  -s [ --source ] arg (=SC_LS_FIXED_V0) Source representation. Currently only 
                                        'SC_LS_FIXED_V0' is supported.
  -f [ --format ] arg (=json)           Output format: 'json' or 'markdown'.
  -i [ --input ] arg                    Input file
  -o [ --output ] arg (=compile_info.json)
                                        Output file


なお、`asm` で出力したファイルについては `profile` はできません。
`profile` は pipeline state の `parameter` / `opt` / `metadata` を使って集計します。

## 1. 4 ケースの pipeline state を生成

In [3]:
dim2_pipeline_path = "data/tutorial_5_dim2_pipeline.yaml"
dim3_pipeline_path = "data/tutorial_5_dim3_pipeline.yaml"
dist_pipeline_path = "data/tutorial_5_dist_pipeline.yaml"
pbc_pipeline_path = "data/tutorial_5_pbc_pipeline.yaml"

!qret compile --verbose --pipeline {dim2_pipeline_path}
!qret compile --verbose --pipeline {dim3_pipeline_path}
!qret compile --verbose --pipeline {dist_pipeline_path}
!qret compile --verbose --pipeline {pbc_pipeline_path}

2026-03-03 11:09:34 - INFO  - Load OpenQASM2.
2026-03-03 11:09:34 - INFO  - Build IR from OpenQASM2.
2026-03-03 11:09:34 - INFO  - Simplify IR before compiling to SC_LS_FIXED_V0.
2026-03-03 11:09:34 - INFO  - Lowering IR to the machine function of SC_LS_FIXED_V0.
2026-03-03 11:09:34 - INFO  - Run passes.
2026-03-03 11:09:34 - INFO  - Run InitCompileInfo
2026-03-03 11:09:34 - INFO  - Initialize compile information
2026-03-03 11:09:34 - INFO  - Run Mapping
2026-03-03 11:09:34 - INFO  - Run Routing
2026-03-03 11:09:34 - INFO  - Save SC_LS_FIXED_V0 pipeline state file.
2026-03-03 11:09:34 - INFO  - Saving SC_LS_FIXED_V0 pipeline state to JSON file: tutorial_5_dim2.json
2026-03-03 11:09:34 - INFO  - Load OpenQASM2.
2026-03-03 11:09:34 - INFO  - Build IR from OpenQASM2.
2026-03-03 11:09:34 - INFO  - Simplify IR before compiling to SC_LS_FIXED_V0.
2026-03-03 11:09:34 - INFO  - Lowering IR to the machine function of SC_LS_FIXED_V0.
2026-03-03 11:09:34 - INFO  - Run passes.
2026-03-03 11:09:34 

## 2. profile を実行

In [4]:
!qret profile -i tutorial_5_dim2.json -o profile_dim2.json
!qret profile -i tutorial_5_dim3.json -o profile_dim3.json
!qret profile -i tutorial_5_dist.json -o profile_dist.json
!qret profile -i tutorial_5_pbc.json -o profile_pbc.json

## 3. 主要指標を表で比較

In [5]:
def summarize(path: str) -> dict:
    data = json.loads(pathlib.Path(path).read_text())
    return {
        "runtime": data.get("runtime"),
        "gate_count": data.get("gate_count"),
        "magic_state_consumption_count": data.get("magic_state_consumption_count"),
        "code_distance": data.get("code_distance"),
        "num_physical_qubits": data.get("num_physical_qubits"),
    }


for name, path in [
    ("Dim2", "profile_dim2.json"),
    ("Dim3", "profile_dim3.json"),
    ("DistributedDim2", "profile_dist.json"),
    ("PBC", "profile_pbc.json"),
]:
    print(f"[{name}]")
    for k, v in summarize(path).items():
        print(f"  {k}: {v}")
    print()

[Dim2]
  runtime: 25
  gate_count: 24
  magic_state_consumption_count: 2
  code_distance: 5
  num_physical_qubits: 4800

[Dim3]
  runtime: 27
  gate_count: 29
  magic_state_consumption_count: 2
  code_distance: 5
  num_physical_qubits: 4800

[DistributedDim2]
  runtime: 35
  gate_count: 27
  magic_state_consumption_count: 2
  code_distance: 5
  num_physical_qubits: 19400

[PBC]
  runtime: 23
  gate_count: 50
  magic_state_consumption_count: 2
  code_distance: 7
  num_physical_qubits: 9408



## 4. PBC の結果 JSON を確認

In [6]:
Code(filename="profile_pbc.json", language="json")

{"use_magic_state_cultivation":false,"magic_factory_seed_offset":0,"magic_generation_period":15,"prob_magic_state_creation":1.0,"maximum_magic_state_stock":10000,"entanglement_generation_period":100,"maximum_entangled_state_stock":10,"reaction_time":1,"topology":[{"type":"plane","coord":[10,10,0],"magic_factory":[{"symbol":0,"coord":[0,0]},{"symbol":1,"coord":[0,1]},{"symbol":2,"coord":[0,2]},{"symbol":3,"coord":[0,3]}]}],"runtime":23,"runtime_without_topology":22,"gate_count":50,"gate_count_detail":{"ALLOCATE":12,"ALLOCATE_MAGIC_FACTORY":4,"DEALLOCATE":12,"MEAS_ZX":9,"TWIST":4,"LATTICE_SURGERY":5,"MOVE_MAGIC":2,"XOR":2},"gate_depth":5,"gate_throughput":[11,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,3,1,3,9,8,2,15],"gate_throughput_ave":2.347826086956522,"gate_throughput_peak":15,"measurement_feedback_count":4,"measurement_feedback_depth":1,"measurement_feedback_rate":[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,2,1,0,0,0],"measurement_feedback_rate_ave":0.17391304347826086,"measurement_feedback_rate_peak":2,"runtime_estimation_measurement_feedback_count":4,"runtime_estimation_measurement_feedback_depth":1,"magic_state_consumption_count":2,"magic_state_consumption_depth":1,"magic_state_consumption_rate":[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,0,0,0,0,0,0],"magic_state_consumption_rate_ave":0.08695652173913043,"magic_state_consumption_rate_peak":2,"runtime_estimation_magic_state_consumption_count":30,"runtime_estimation_magic_state_consumption_depth":15,"magic_factory_count":4,"entanglement_consumption_count":0,"entanglement_consumption_depth":0,"entanglement_consumption_rate":[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],"entanglement_consumption_rate_ave":0.0,"entanglement_consumption_rate_peak":0,"runtime_estimation_entanglement_consumption_count":0,"runtime_estimation_entanglement_consumption_depth":0,"entanglement_factory_count":0,"chip_cell_count":96,"chip_cell_algorithmic_qubit":[5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,6,6,6,5,2,1,0],"chip_cell_algorithmic_qubit_ave":4.608695652173913,"chip_cell_algorithmic_qubit_peak":6,"chip_cell_algorithmic_qubit_ratio":[0.052083333333333336,0.052083333333333336,0.052083333333333336,0.052083333333333336,0.052083333333333336,0.052083333333333336,0.052083333333333336,0.052083333333333336,0.052083333333333336,0.052083333333333336,0.052083333333333336,0.052083333333333336,0.052083333333333336,0.052083333333333336,0.052083333333333336,0.052083333333333336,0.0625,0.0625,0.0625,0.052083333333333336,0.020833333333333332,0.010416666666666666,0.0],"chip_cell_algorithmic_qubit_ratio_ave":0.0480072463768116,"chip_cell_algorithmic_qubit_ratio_peak":0.0625,"chip_cell_active_qubit_area":[7,7,5,5,5,5,5,5,5,5,5,5,5,5,5,5,24,12,19,14,4,2,0],"chip_cell_active_qubit_area_ave":6.913043478260869,"chip_cell_active_qubit_area_peak":24,"chip_cell_active_qubit_area_ratio":[0.07291666666666667,0.07291666666666667,0.052083333333333336,0.052083333333333336,0.052083333333333336,0.052083333333333336,0.052083333333333336,0.052083333333333336,0.052083333333333336,0.052083333333333336,0.052083333333333336,0.052083333333333336,0.052083333333333336,0.052083333333333336,0.052083333333333336,0.052083333333333336,0.25,0.125,0.19791666666666666,0.14583333333333334,0.041666666666666664,0.020833333333333332,0.0],"chip_cell_active_qubit_area_ratio_ave":0.0720108695652174,"chip_cell_active_qubit_area_ratio_peak":0.25,"qubit_volume":159,"code_distance":7,"execution_time_sec":1.61e-06,"num_physical_qubits":9408}

## visualize_compile_info で比較する
`profile_*.json` を GUI で比較したい場合は、`visualize_compile_info.py` を起動します。

```sh
streamlit run ../../visualizer/visualize_compile_info.py
```

主なタブ:
- `Overview`: 主要指標の比較、`Baseline` 指定で差分確認
- `Tables`: 詳細テーブル（`gate_count_detail` が無い JSON では Details は空）
- `Time Series`: シリーズ選択と beat 範囲指定
- `Topology`: トポロジー footprint 比較
